## Particles, densities and principles

In the [dynamic transport section](/aiml-common/lectures/optimal-transport/dynamic-transport/index) every particle followed a velocity field, and its path was smooth. In the [Brownian motion section](/aiml-common/lectures/diffusion/brownian-motion/brownian_motion) particles moved at random, and the density still changed smoothly in time. This section asks whether deterministic dynamics can reproduce the densities of a random process.

The answer needs three levels of description kept apart.

| Level | Object | Question |
|---|---|---|
| Particles | SDE or ODE | How does one particle move? |
| Density | Continuity or Fokker-Planck equation | How does the population evolve? |
| Principle | Action or free energy | Why does the population follow this path? |

This section works at the first two levels and ends with a measurement at the third: the kinetic action of the density path. The next section supplies the principle that diffusion follows.

## The running example

The data distribution $p_0$ is the Gaussian mixture of the probability transport chapter, with weights $\pi_k = (0.3, 0.5, 0.2)$, means $\mu_k = (-2, 1, 3.5)$ and standard deviations $\sigma_k = (0.4, 0.6, 0.3)$. The forward process is the Ornstein-Uhlenbeck process of the Brownian motion section,

$$
dX_t = -X_t\,dt + \sqrt{2}\,dW_t ,
$$

so the drift is $f(x) = -x$, the noise level is $g = \sqrt{2}$ and the diffusion coefficient is $D = g^2/2 = 1$.

At each fixed $t$, $X_t$ has the same distribution as $e^{-t} X_0 + \sqrt{1 - e^{-2t}}\,Z$ with $Z \sim \mathcal{N}(0, 1)$ independent of $X_0$. The density $p_t$ is therefore again a mixture of three Gaussians. Component $k$ keeps its weight, has mean $\mu_k e^{-t}$ and has variance $\sigma_k^2 e^{-2t} + 1 - e^{-2t}$. This statement holds in distribution at each time separately, so paths below are built step by step, as in the drift part of the [Brownian motion section](/aiml-common/lectures/diffusion/brownian-motion/brownian_motion).

Because $p_t$ is a mixture in closed form, so are its time derivative and its score $\partial_x \log p_t$. The quantile function $F_t^{-1}$ comes from bisection on the closed-form CDF. Everything else on this page is numerical, and each place says how it was computed: particle paths come from integrator steps, distances between samples and densities from Monte Carlo, and actions from quadrature on a grid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm

# p_0 is the phase 1 Gaussian mixture. The forward process is dX = -X dt + sqrt(2) dW, so D = 1.
PI = np.array([0.3, 0.5, 0.2])
MU = np.array([-2.0, 1.0, 3.5])
SIGMA = np.array([0.4, 0.6, 0.3])
U_CLIP = 1e-12  # quantile levels are clipped to [U_CLIP, 1 - U_CLIP]


def mixture_pdf(x, pi=PI, mu=MU, sigma=SIGMA):
    x = np.asarray(x, dtype=float)[..., None]
    return np.sum(pi * norm.pdf(x, mu, sigma), axis=-1)


def mixture_cdf(x, pi=PI, mu=MU, sigma=SIGMA):
    x = np.asarray(x, dtype=float)[..., None]
    return np.sum(pi * norm.cdf(x, mu, sigma), axis=-1)


def mixture_ppf(u, pi=PI, mu=MU, sigma=SIGMA, iters=64):
    """Quantile function F_1^{-1}(u) by vectorized bisection on the mixture CDF.

    The root lies between the smallest and the largest component quantile at level u.
    """
    u = np.clip(np.asarray(u, dtype=float), U_CLIP, 1 - U_CLIP)
    q = mu + sigma * norm.ppf(u)[..., None]
    lo, hi = q.min(axis=-1), q.max(axis=-1)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        below = mixture_cdf(mid, pi, mu, sigma) < u
        lo = np.where(below, mid, lo)
        hi = np.where(below, hi, mid)
    return 0.5 * (lo + hi)


def ou_components(t, pi=PI, mu=MU, sigma=SIGMA):
    """Weights, means mu e^{-t} and standard deviations of the components of p_t."""
    pi, mu, sigma = (np.asarray(a, dtype=float) for a in (pi, mu, sigma))
    decay = np.exp(-t)
    return pi, mu * decay, np.sqrt(sigma**2 * decay**2 + 1 - decay**2)


def p_t(x, t, **mix):
    """Density of the Ornstein-Uhlenbeck process at time t, started from the mixture."""
    return mixture_pdf(x, *ou_components(t, **mix))


def dp_dt(x, t, **mix):
    """Analytic time derivative of p_t, through the moving component means and variances."""
    pi, m, s = ou_components(t, **mix)
    sigma = np.asarray(mix.get("sigma", SIGMA), dtype=float)
    x = np.asarray(x, dtype=float)[..., None]
    z = (x - m) / s
    dens = pi * norm.pdf(x, m, s)
    dm = -m                                       # d/dt of mu e^{-t}
    dvar = 2 * np.exp(-2 * t) * (1 - sigma**2)    # d/dt of sigma^2 e^{-2t} + 1 - e^{-2t}
    return np.sum(dens * (z / s * dm + 0.5 * (z**2 - 1) / s**2 * dvar), axis=-1)


def score(x, t, **mix):
    """d/dx log p_t, a responsibility-weighted sum of component scores (log-sum-exp weights).

    Never computed as (dp/dx) / p, so it stays finite far in the tails.
    """
    pi, m, s = ou_components(t, **mix)
    x = np.asarray(x, dtype=float)[..., None]
    log_w = np.log(pi) + norm.logpdf(x, m, s)
    resp = np.exp(log_w - logsumexp(log_w, axis=-1, keepdims=True))
    return np.sum(resp * (-(x - m) / s**2), axis=-1)


def velocity(x, t, **mix):
    """Probability flow velocity v_t(x) = f(x) - D d/dx log p_t(x) with f(x) = -x, D = 1."""
    return -np.asarray(x, dtype=float) - score(x, t, **mix)


def quantile_t(u, t, **mix):
    """F_t^{-1}(u), the quantile function of p_t."""
    return mixture_ppf(u, *ou_components(t, **mix))


def w2_squared_between(qa, qb):
    """Mean of squared quantile differences, for quantiles on a midpoint u grid."""
    return float(np.mean((np.asarray(qa) - np.asarray(qb)) ** 2))


def simulate(x0, lam, t_grid, rng, substeps=20, **mix):
    """Paths of dX = [-X + (lam^2 - 1) score] dt + lam sqrt(2) dW from x0, shape (len(t_grid), len(x0)).

    lam = 1 uses the exact Ornstein-Uhlenbeck transition between grid times. Other values of
    lam use Euler-Maruyama with the exact score and `substeps` steps between grid times.
    """
    x = np.array(x0, dtype=float)
    out = np.empty((len(t_grid), len(x)))
    out[0] = x
    for i in range(1, len(t_grid)):
        t0, t1 = t_grid[i - 1], t_grid[i]
        if lam == 1:
            h = t1 - t0
            x = np.exp(-h) * x + np.sqrt(1 - np.exp(-2 * h)) * rng.standard_normal(len(x))
        else:
            h = (t1 - t0) / substeps
            for k in range(substeps):
                drift = -x + (lam**2 - 1) * score(x, t0 + k * h, **mix)
                x = x + drift * h + lam * np.sqrt(2 * h) * rng.standard_normal(len(x))
        out[i] = x
    return out


# Horizon T from the contraction bound W_2(p_t, N(0, 1)) <= e^{-t} W_2(p_0, N(0, 1)).
U_MID = (np.arange(20000) + 0.5) / 20000   # midpoint grid on (0, 1)


def w2_to_gaussian(t, **mix):
    """W_2(p_t, N(0, 1)) from quantile functions."""
    return np.sqrt(w2_squared_between(quantile_t(U_MID, t, **mix), norm.ppf(U_MID)))


def bound(t, **mix):
    """Contraction bound e^{-t} W_2(p_0, N(0, 1))."""
    return np.exp(-t) * w2_to_gaussian(0.0, **mix)


T_TOL = 0.05
T = float(next(n for n in range(1, 50) if bound(n) < T_TOL))   # smallest whole-number horizon

In [ ]:
def plot_transport(ax, t_grid, x_grid, density, trajectories=None, title="",
                   action=None, edge=0.22):
    """Standard figure: heatmap of p_t(x) with time on the horizontal axis.

    density has shape (len(x_grid), len(t_grid)). The source and target columns are
    drawn as profiles outside the left and right edges. trajectories, if given, has
    shape (len(t_grid), n_particles). action, if given, is reported in the title.
    """
    vmax = np.quantile(density, 0.995)
    ax.pcolormesh(t_grid, x_grid, density, shading="auto", cmap="magma", vmin=0, vmax=vmax)
    for col, side, color in ((0, -1, "tab:blue"), (-1, 1, "tab:orange")):
        profile = density[:, col] / density[:, col].max()
        t_edge = t_grid[col]
        ax.fill_betweenx(x_grid, t_edge, t_edge + side * edge * profile,
                         color=color, alpha=0.7, lw=0)
    if trajectories is not None:
        ax.plot(t_grid, trajectories, color="white", lw=0.6, alpha=0.7)
    ax.set_xlim(t_grid[0] - 1.05 * edge, t_grid[-1] + 1.05 * edge)
    ax.set_ylim(x_grid[0], x_grid[-1])
    ax.set_xticks(np.linspace(t_grid[0], t_grid[-1], 5))
    ax.set_xlabel("t")
    ax.set_ylabel("x")
    if action is not None:
        title = f"{title} (kinetic action {action:.3f})" if title else f"kinetic action {action:.3f}"
    ax.set_title(title)

## The horizon

The Ornstein-Uhlenbeck process reaches $\mathcal{N}(0, 1)$ only as $t \to \infty$. Its distance to that limit shrinks at a known rate:

$$
W_2\bigl(p_t, \mathcal{N}(0, 1)\bigr) \le e^{-t}\, W_2\bigl(p_0, \mathcal{N}(0, 1)\bigr).
$$

The horizon $T$ is the smallest whole number at which this bound falls below $0.05$. The bound and the computed distance at each whole $t$ up to $T$ follow.

In [ ]:
print(f"W_2(p_0, N(0, 1)) = {w2_to_gaussian(0.0):.4f}")
for n in range(1, int(T) + 1):
    print(f"t = {n}: bound {bound(n):.4f}   computed W_2(p_t, N(0, 1)) {w2_to_gaussian(n):.4f}")
print(f"tolerance {T_TOL}, horizon T = {T:g}")

In [ ]:
assert np.all(np.isfinite(score(np.linspace(-12, 12, 4001), 0.0)))
assert np.all(np.isfinite(quantile_t(np.array([0.0, 1.0]), 0.0)))   # clipped quantile ends
# T from the contraction bound, and the computed W2 under it
assert w2_to_gaussian(T) <= bound(T) + 1e-9 and bound(T) < T_TOL

The bound first drops below $0.05$ at $T = 4$, where the computed distance is smaller still. The endpoint $p_T$ is close to $\mathcal{N}(0, 1)$ but not equal to it, and this page always uses the true $p_T$.

## Diffusion time and transport time

This page uses the diffusion convention: data at $t = 0$ and noise at $t = T$. The probability transport chapter ran the other way, from noise at $s = 0$ to data at $s = 1$. The two clocks are related by

$$
s = 1 - t/T .
$$

The transport chapter's $p_0$, the noise, is $p_T$ here, and its $p_1$, the data, is $p_0$ here. Every figure on this page puts data on the left and noise on the right, so the transport chapter's figures read in the opposite direction.

The DDPM and DDIM sections use the variance-preserving SDE $dX = -\tfrac12 \beta(t)\,X\,dt + \sqrt{\beta(t)}\,dW$. With the clock $\theta(t) = \tfrac12 \int_0^t \beta(r)\,dr$ it becomes the Ornstein-Uhlenbeck process used here, so every result on this page carries over to those sections after a change of clock.

## The Fokker-Planck equation

Conservation of mass gives an equation for $p_t$ by the interval argument of the dynamic transport section. Let $J_t(x)$ be the rate at which mass crosses $x$ from left to right. The mass in $[a, b]$ changes only through the two ends, so

$$
\frac{d}{dt}\int_a^b p_t\,dx = J_t(a) - J_t(b) = -\int_a^b \partial_x J_t\,dx .
$$

This holds for every interval, so $\partial_t p_t = -\partial_x J_t$ at every point. Conservation alone says nothing about what $J_t$ is. The Brownian motion section supplied that: the drift carries mass with flux $f\,p$, and unbiased random steps carry mass down the gradient with the Fick flux $-D\,\partial_x p$. Their sum gives the Fokker-Planck equation,

$$
\partial_t p_t = -\partial_x (f\,p_t) + D\,\partial_x^2 p_t .
$$

For the Ornstein-Uhlenbeck process it reads $\partial_t p_t = \partial_x(x\,p_t) + \partial_x^2 p_t$.

The closed-form $p_t$ should satisfy this equation. On the left, the analytic time derivative is taken through the moving means and variances of the components. On the right, the $x$-derivatives are analytic too. The integrated residual $\int |\partial_t p_t - \text{right side}|\,dx$ and the total mass $\int p_t\,dx$ are computed on $[-12, 12]$, at $t = 0.1, 0.5, 1, 2$ and $T$.

In [ ]:
X_FP = np.linspace(-12, 12, 24001)
T_CHECK = (0.1, 0.5, 1.0, 2.0)


def fp_rhs(x, t, **mix):
    """-d/dx(f p_t) + D d^2/dx^2 p_t with f(x) = -x and D = 1, from analytic x-derivatives."""
    pi, m, s = ou_components(t, **mix)
    xx = np.asarray(x, dtype=float)[..., None]
    z = (xx - m) / s
    dens = pi * norm.pdf(xx, m, s)
    p = dens.sum(axis=-1)
    p_x = np.sum(-dens * z / s, axis=-1)
    p_xx = np.sum(dens * (z**2 - 1) / s**2, axis=-1)
    return p + x * p_x + p_xx          # -d/dx(-x p) = p + x p'


def fp_residual_l1(t, **mix):
    """Integral of |dp_t/dt - right-hand side| over x."""
    return np.trapezoid(np.abs(dp_dt(X_FP, t, **mix) - fp_rhs(X_FP, t, **mix)), X_FP)


def mass(t, **mix):
    return np.trapezoid(p_t(X_FP, t, **mix), X_FP)


for t in T_CHECK + (T,):
    print(f"t = {t:3.1f}   integrated residual {fp_residual_l1(t):.1e}   mass {mass(t):.8f}")

In [ ]:
# Fokker-Planck: integrated residual and mass, analytic dp/dt
assert max(fp_residual_l1(t) for t in T_CHECK + (T,)) < 1e-6
assert max(abs(mass(t) - 1) for t in T_CHECK + (T,)) < 1e-6

The residual is at the level of rounding error at every time, so the closed form of $p_t$ solves the Fokker-Planck equation. The mass stays 1 to eight digits.

## Drift, score and probability flow velocity

Pull $p_t$ out of the total flux:

$$
\begin{aligned}
J_t &= f\,p_t - D\,\partial_x p_t \\
&= p_t\,\bigl(f - D\,\partial_x \log p_t\bigr) .
\end{aligned}
$$

The flux now has the form $p_t\,v_t$ of the continuity equation, with the velocity

$$
v_t(x) = f(x) - D\,\partial_x \log p_t(x) = -x - \partial_x \log p_t(x) .
$$

The Fokker-Planck equation is the continuity equation $\partial_t p_t + \partial_x(p_t\,v_t) = 0$ for this field. Particles that follow the ordinary differential equation $\dot x = v_t(x)$ from samples of $p_0$ therefore have density $p_t$ at every time, the same as the particles of the SDE. This ODE is the probability flow ODE [@Song2020-ko].

Three different quantities appear here.

| Quantity | Ornstein-Uhlenbeck | Meaning |
|---|---|---|
| Drift $f$ | $-x$ | mean displacement of a particle per unit time |
| Score $\partial_x \log p_t$ | depends on $p_t$ | direction in which log density increases |
| Velocity $v_t$ | $-x - \partial_x \log p_t$ | velocity that carries the density |

In deterministic transport the drift and the velocity are the same thing. With noise they differ by $D$ times the score. For a mixture, the score is a weighted sum of the component scores, with weights equal to the probability that $x$ came from each component. Computed this way it stays finite far in the tails, where $p_t$ itself is too small to divide by.

The same velocity has a second description. At each fixed time, $X_t$ has the distribution of $a_t X_0 + b_t Z$ with $a_t = e^{-t}$ and $b_t = \sqrt{1 - e^{-2t}}$. A point on this interpolation moves with velocity $\dot a_t X_0 + \dot b_t Z$. Average that velocity over all pairs $(X_0, Z)$ that pass through $x$ at time $t$:

$$
u_t(x) = \mathbb{E}\bigl[\dot a_t X_0 + \dot b_t Z \mid X_t = x\bigr] .
$$

Within one component, $X_0$, $Z$ and $X_t$ are jointly Gaussian, so the conditional means of $X_0$ and $Z$ are linear in $x$. Weighting them by the component probabilities gives $u_t(x)$ in closed form. Below, $u_t$ is compared with $-x - \partial_x \log p_t$ at $t = 0.1, 0.5, 1, 2$ and $T$, wherever $p_t > 10^{-6}$.

In [ ]:
xs = np.linspace(-12, 12, 4001)


def conditional_mean_field(x, t, **mix):
    """E[a'_t X_0 + b'_t Z | X_t = x] for X_t = a_t X_0 + b_t Z, a_t = e^{-t}, b_t = sqrt(1 - e^{-2t}).

    Given component k, X_0 and Z are jointly Gaussian with X_t, so their posterior means are
    linear in x. The responsibilities weight the per-component means.
    """
    pi, mu, sigma = (np.asarray(mix.get(k, d), dtype=float)
                     for k, d in (("pi", PI), ("mu", MU), ("sigma", SIGMA)))
    a, b = np.exp(-t), np.sqrt(1 - np.exp(-2 * t))
    da, db = -a, a**2 / b                               # time derivatives of a_t and b_t
    x = np.asarray(x, dtype=float)[..., None]
    var = a**2 * sigma**2 + b**2                       # variance of X_t given component k
    log_w = np.log(pi) + norm.logpdf(x, a * mu, np.sqrt(var))
    resp = np.exp(log_w - logsumexp(log_w, axis=-1, keepdims=True))
    mean_x0 = mu + a * sigma**2 / var * (x - a * mu)    # E[X_0 | X_t = x, k]
    mean_z = b / var * (x - a * mu)                     # E[Z | X_t = x, k]
    return np.sum(resp * (da * mean_x0 + db * mean_z), axis=-1)


fm_gap = max(np.max(np.abs(conditional_mean_field(xs, t) - velocity(xs, t))[p_t(xs, t) > 1e-6])
             for t in T_CHECK + (T,))
print(f"largest |conditional-mean field - (-x - score)| where p_t > 1e-6: {fm_gap:.1e}")

In [ ]:
# the probability flow velocity is the flow matching conditional-mean field of this path
assert fm_gap < 1e-8

The two fields agree to rounding error. The probability flow velocity of this path is the conditional-mean field that flow matching regresses: the expected velocity $\dot a_t X_0 + \dot b_t Z$ of the interpolation $X_t = a_t X_0 + b_t Z$, given that it passes through $x$. Flow matching and stochastic interpolants reach the same field from the interpolation, without the Fokker-Planck equation [@Albergo2023-zq].

## Equilibrium

The simplest case shows the difference between drift and velocity. For $p_\infty = \mathcal{N}(0, 1)$ the score is $\partial_x \log p_\infty = -x$, so

$$
v_\infty(x) = -x - (-x) = 0 .
$$

The density stands still while each particle keeps moving under $dX = -X\,dt + \sqrt{2}\,dW$. At every point the drift flux $-x\,p_\infty$, which points toward the origin, cancels the diffusive flux $-\partial_x p_\infty = x\,p_\infty$, which points away from it. The rotation field of the [velocity fields section](/aiml-common/lectures/velocity-fields/index) is the deterministic analogue: there too the particles move while a rotationally symmetric density stays still. Two quantities confirm this: the general score function evaluated on $\mathcal{N}(0, 1)$, and the velocity of the mixture path at $t = 40$, where $p_t$ equals $\mathcal{N}(0, 1)$ to within $e^{-40}$.

In [ ]:
def gaussian_score(x):
    """Score of N(0, 1), computed by the general score function with one component."""
    return score(x, 0.0, pi=[1.0], mu=[0.0], sigma=[1.0])


print(f"largest |f(x) - score(x)| for N(0, 1), |x| <= 12: {np.max(np.abs(-xs - gaussian_score(xs))):.1e}")
print(f"largest |v_t(x)| of the mixture path at t = 40:    {np.max(np.abs(velocity(xs, 40.0))):.1e}")

In [ ]:
# equilibrium
assert np.max(np.abs(-xs - gaussian_score(xs))) < 1e-12
assert np.max(np.abs(velocity(xs, 40.0))) < 1e-12

In [ ]:
x_eq = np.linspace(-4, 4, 401)
p_eq = norm.pdf(x_eq)
drift_flux = -x_eq * p_eq           # f p with f(x) = -x
diff_flux = x_eq * p_eq             # -D dp/dx with D = 1
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(x_eq, drift_flux, color="tab:blue", lw=2, label=r"drift flux $f\,p = -x\,p$")
ax.plot(x_eq, diff_flux, color="tab:orange", lw=2, label=r"diffusive flux $-D\,\partial_x p = x\,p$")
ax.plot(x_eq, drift_flux + diff_flux, color="black", lw=1.5, ls="--", label="total flux")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("x")
ax.set_ylabel("flux")
ax.set_title(r"Fluxes at equilibrium, $p = \mathcal{N}(0, 1)$")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

*The two fluxes of $\mathcal{N}(0, 1)$ and their sum. The drift pushes mass inward and diffusion spreads it outward at the same rate at every $x$, so the total flux is zero and the density does not change.*

## Same marginals, different couplings

The SDE and the probability flow ODE are two members of a family. For every $\lambda \ge 0$,

$$
\begin{aligned}
dX_t &= \Bigl[f + \tfrac{\lambda^2 - 1}{2}\,g^2\,\partial_x \log p_t\Bigr] dt \\
&\quad + \lambda\,g\,dW_t .
\end{aligned}
$$

Its drift flux is $\bigl[f + (\lambda^2 - 1) D\,\partial_x \log p_t\bigr] p_t$ and its diffusive flux is $-\lambda^2 D\,\partial_x p_t$. Since $p_t\,\partial_x \log p_t = \partial_x p_t$, they add up to

$$
J_t = f\,p_t - D\,\partial_x p_t
$$

for every $\lambda$. The flux is the same, so the Fokker-Planck equation is the same, and every member started from $p_0$ has the same density $p_t$ at every time. The value $\lambda = 1$ gives the forward SDE and $\lambda = 0$ gives the probability flow ODE.

The simulation starts 2000 samples of $p_0$ and moves the same samples under $\lambda = 0$, $0.5$ and $1$. For $\lambda = 1$ each step uses the exact Ornstein-Uhlenbeck transition. For the other two, each step is an Euler-Maruyama step of length $5 \times 10^{-4}$ with the exact score. If the marginals agree, the empirical $W_1$ distance from the particles to the closed-form $p_t$ should be about as small as the $W_1$ distance of 2000 independent exact samples of $p_t$. That second distance, averaged over 20 draws, is the sampling baseline: the distance that sampling alone produces.

In [ ]:
LAMBDAS = (0.0, 0.5, 1.0)
N_PART = 2000
T_SIM = np.linspace(0, T, 401)
X_W1 = np.linspace(-12, 12, 24001)


def sample_p_t(n, t, rng, **mix):
    """Independent exact samples of p_t: pick a component, then draw from it."""
    pi, m, s = ou_components(t, **mix)
    k = rng.choice(len(pi), size=n, p=pi)
    return m[k] + s[k] * rng.standard_normal(n)


def w1_to_exact(samples, t, **mix):
    """W_1 between the empirical distribution of samples and p_t, as the integral of |F_emp - F_t|."""
    emp = np.searchsorted(np.sort(samples), X_W1, side="right") / len(samples)
    return np.trapezoid(np.abs(emp - mixture_cdf(X_W1, *ou_components(t, **mix))), X_W1)


rng = np.random.default_rng(0)
x0 = sample_p_t(N_PART, 0.0, rng)
paths = {lam: simulate(x0, lam, T_SIM, rng) for lam in LAMBDAS}

t_check_all = T_CHECK + (T,)
idx_check = [int(np.argmin(np.abs(T_SIM - t))) for t in t_check_all]
w1_table = np.array([[w1_to_exact(paths[lam][i], t) for i, t in zip(idx_check, t_check_all)]
                     for lam in LAMBDAS])
# Sampling baseline: mean W_1 of independent exact samples of the same size, 20 repetitions
w1_baseline = np.array([np.mean([w1_to_exact(sample_p_t(N_PART, t, rng), t) for _ in range(20)])
                        for t in t_check_all])
xT_lam0, xT_lam1 = paths[0.0][-1], paths[1.0][-1]
print("largest W_1 to the exact p_t over all lambda and check times: "
      f"{w1_table.max():.4f} (largest sampling baseline {w1_baseline.max():.4f})")

In [ ]:
# same marginals: empirical W1 within 3x the exact-sample baseline, every lambda, every check time
assert np.all(w1_table <= 3 * w1_baseline)
# couplings differ: lambda = 0 coupling is monotone, lambda = 1 is not
assert np.all(np.diff(xT_lam0[np.argsort(x0)]) >= -1e-9) and np.any(np.diff(xT_lam1[np.argsort(x0)]) < 0)

In [ ]:
x_plot = np.linspace(-4.5, 5.5, 401)
p_grid = np.stack([p_t(x_plot, t) for t in T_SIM], axis=1)
show = np.argsort(x0)[np.linspace(40, N_PART - 41, 25).astype(int)]   # 25 particles, spread over p_0
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, lam in zip(axes, LAMBDAS):
    plot_transport(ax, T_SIM, x_plot, p_grid, trajectories=paths[lam][:, show],
                   title=fr"$\lambda = {lam:g}$")
for ax in axes[1:]:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

*The same density path under three dynamics. Each panel shows $p_t$ as a heatmap and 25 particle paths in white. At $\lambda = 0$ the paths are smooth and never cross. At $\lambda = 1$ they are Brownian. In all three panels the particles fill the same density.*

The densities agree, and the particles still end up in different places. A scatter of the starting point $X_0$ against the endpoint $X_T$ shows the coupling that each dynamics produces.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, lam in zip(axes, LAMBDAS):
    ax.scatter(x0, paths[lam][-1], s=3, alpha=0.4, color="tab:purple")
    ax.set_title(fr"$\lambda = {lam:g}$")
    ax.set_xlabel("$X_0$")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("$X_T$")
plt.tight_layout()
plt.show()

*Starting point against endpoint for each $\lambda$. At $\lambda = 0$ the endpoint is an increasing function of the starting point. At $\lambda = 0.5$ the endpoint still depends on the starting point, but the coupling spreads widely. At $\lambda = 1$ the endpoint is nearly independent of $X_0$.*

All three dynamics have the same two marginals $p_0$ and $p_T$, and the couplings are different. The probability flow gives a deterministic, increasing map. The forward SDE gives a coupling close to independence, since four time units of noise have erased most of the starting position. This is the point of the [maps and couplings section](/aiml-common/lectures/optimal-transport/maps-and-couplings/index): the marginals do not fix the coupling. A later chapter on Schrödinger bridges picks one coupling among the stochastic ones by a principle of its own.

## Running the flow backward

The probability flow ODE is deterministic, so it can be run from $t = T$ back to $t = 0$. Start from samples of $p_T$, integrate $\dot x = v_t(x)$ backward with 400 fourth-order Runge-Kutta steps, and the samples should arrive distributed as the mixture $p_0$. This is the deterministic sampler of diffusion models. A generative model starts from $\mathcal{N}(0, 1)$ instead of $p_T$ and learns the score instead of computing it; here both are exact, so only the integrator can introduce error.

The [DDIM section](/aiml-common/lectures/diffusion/ddim/ddim_mog_tutorial) uses the same idea. Deterministic DDIM ($\eta = 0$) is a discretization of the probability flow ODE of the variance-preserving SDE, in the clock $\theta$ defined in the section on diffusion time.

In [ ]:
N_STEPS = 400   # fourth-order Runge-Kutta steps over [0, T]


def pf_solve(x, t_start, t_end, steps=N_STEPS, **mix):
    """Integrate dx/dt = v_t(x) from t_start to t_end (either direction), fourth-order Runge-Kutta."""
    x = np.array(x, dtype=float)
    h = (t_end - t_start) / steps
    for k in range(steps):
        t = t_start + k * h
        k1 = velocity(x, t, **mix)
        k2 = velocity(x + 0.5 * h * k1, t + 0.5 * h, **mix)
        k3 = velocity(x + 0.5 * h * k2, t + 0.5 * h, **mix)
        k4 = velocity(x + h * k3, t + h, **mix)
        x = x + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return x


xT_samples = sample_p_t(N_PART, T, rng)
x0_recovered = pf_solve(xT_samples, T, 0.0)
w1_recovered = w1_to_exact(x0_recovered, 0.0)
w1_baseline_p0 = np.mean([w1_to_exact(sample_p_t(N_PART, 0.0, rng), 0.0) for _ in range(20)])
print(f"W_1(recovered samples, p_0) = {w1_recovered:.4f}   sampling baseline {w1_baseline_p0:.4f}")

In [ ]:
# backward ODE recovers p_0
assert w1_recovered <= 3 * w1_baseline_p0

The recovered samples are within three times the sampling baseline of $p_0$. The integration error, measured below on the endpoint map, is about $10^{-6}$. The remaining distance is consistent with the sampling noise of the 2000 starting points, carried through the map.

In [ ]:
x_fine = np.linspace(-4.5, 5.5, 801)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
axes[0].hist(xT_samples, bins=60, density=True, color="tab:orange", alpha=0.5, label="samples of $p_T$")
axes[0].plot(x_fine, p_t(x_fine, T), color="black", lw=1.2, label="$p_T$")
axes[0].set_title(f"start, t = {T:g}")
axes[1].hist(x0_recovered, bins=60, density=True, color="tab:blue", alpha=0.5,
             label="after the backward ODE")
axes[1].plot(x_fine, p_t(x_fine, 0.0), color="black", lw=1.2, label="$p_0$")
axes[1].set_title("end, t = 0")
for ax in axes:
    ax.set_xlabel("x")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
plt.tight_layout()
plt.show()

*Left, samples of $p_T$. Right, the same samples after the backward probability flow, with the mixture $p_0$ drawn in black.*

## Kinetic action of the density path

The dynamic transport section measured a path by its kinetic action. On this page the action belongs to the density path $p_t$ and its velocity $v_t$. It does not belong to Brownian trajectories, which are nowhere differentiable. The discrete estimate $\mathbb{E}\bigl[(\Delta X/\Delta t)^2\bigr]$ for the forward SDE grows like $g^2/\Delta t$, without limit as the step shrinks. One exact step of the forward SDE, started from samples of $p_t$ at $t = 1$, shows the growth.

In [ ]:
dt = 0.01


def brownian_speed_sq(h, n=200000, t=1.0, seed=0):
    """Mean of (Delta X / h)^2 over one exact step of length h of the forward SDE, from p_t."""
    rng_b = np.random.default_rng(seed)
    x = sample_p_t(n, t, rng_b)
    x_next = simulate(x, 1.0, np.array([t, t + h]), rng_b)[-1]
    return np.mean(((x_next - x) / h) ** 2)


for h in (dt, dt / 4):
    print(f"step {h:.4f}: mean squared speed {brownian_speed_sq(h):8.1f}   (g^2 / step = {2 / h:8.1f})")

In [ ]:
# Brownian path "action" grows like 1/dt
assert 3.2 < brownian_speed_sq(dt / 4) / brownian_speed_sq(dt) < 4.8

Quartering the step quadruples the mean squared speed, close to $g^2/\Delta t$ in both cases. The density path has a finite action. In physical time it is

$$
\mathcal{A} = \int_0^T \!\! \int p_t(x)\,v_t(x)^2\,dx\,dt .
$$

To compare it with the transport chapter, change to the normalized time $s = 1 - t/T$. The velocity becomes $-T\,v_t$, with the sign reversed because $s$ runs against $t$, and $|ds| = dt/T$, so the action in normalized time is $\bar{\mathcal{A}} = T\,\mathcal{A}$. The Benamou-Brenier formula says that no path from $p_0$ to $p_T$ has a smaller normalized action than $W_2^2(p_0, p_T)$ [@Benamou2000-gb], so $\bar{\mathcal{A}} \ge W_2^2$.

### The endpoint map is the optimal map

The probability flow keeps particles in order, so its endpoint map is the increasing map $F_T^{-1} \circ F_0$. The [1D mixture section](/aiml-common/lectures/flow-matching/mog-1d/index) showed why an order-preserving flow ends at the optimal map on the line. Below, the flow is integrated forward from 99 quantile levels $u$ in $[0.01, 0.99]$, the endpoints are compared with $F_T^{-1}(u)$, and the integration is repeated with half the step.

In [ ]:
u_mid = np.linspace(0.01, 0.99, 99)


def pf_endpoint(u, steps=N_STEPS, **mix):
    """Probability flow endpoint X_T of the particle that starts at the quantile F_0^{-1}(u)."""
    return pf_solve(quantile_t(u, 0.0, **mix), 0.0, T, steps, **mix)


map_error = np.max(np.abs(pf_endpoint(u_mid) - quantile_t(u_mid, T)))
step_change = np.max(np.abs(pf_endpoint(u_mid, steps=2 * N_STEPS) - pf_endpoint(u_mid)))
print(f"largest |ODE endpoint - F_T^(-1)(F_0(x_0))|: {map_error:.1e}")
print(f"largest change when the step is halved:     {step_change:.1e}")

In [ ]:
# endpoint map is the monotone (optimal) map, and is step-size stable
assert np.max(np.abs(pf_endpoint(u_mid) - quantile_t(u_mid, T))) < 1e-4   # u_mid in [0.01, 0.99]
assert np.max(np.abs(pf_endpoint(u_mid, steps=2 * N_STEPS) - pf_endpoint(u_mid))) < 1e-4

The flow reaches the optimal map to about $10^{-6}$. It reaches it along a path that is not the geodesic, and the next part measures how much that path costs.

### The excess, exactly

Write $Q_t = F_t^{-1}$ for the quantile function of $p_t$. The particle at level $u$ sits at $Q_t(u)$, so $\partial_t Q_t(u) = v_t(Q_t(u))$. The change of variable $x = Q_t(u)$ turns $p_t\,dx$ into $du$, and the action becomes

$$
\mathcal{A} = \int_0^T \!\! \int_0^1 \bigl|\partial_t Q_t(u)\bigr|^2\,du\,dt .
$$

The geodesic between $p_0$ and $p_T$ moves every quantile at constant speed, $Q_t^{\text{geo}} = (1 - t/T)\,Q_0 + (t/T)\,Q_T$, and $W_2^2 = \int_0^1 |Q_T - Q_0|^2\,du$ [@Santambrogio2015-za]. Expand the square below and use $\int_0^T \partial_t Q_t\,dt = Q_T - Q_0$:

$$
\begin{aligned}
\bar{\mathcal{A}} - W_2^2
&= T \int_0^T \!\! \int_0^1 \Bigl|\partial_t Q_t - \frac{Q_T - Q_0}{T}\Bigr|^2 du\,dt .
\end{aligned}
$$

The excess action is the mean squared gap between the quantile speed and the constant geodesic speed. It is zero only when every quantile moves at constant speed.

Numerically, $Q_t(u)$ is evaluated on 1000 midpoint levels in $[0.001, 0.999]$ and 801 times, spaced more densely near $t = 0$, and $\partial_t Q_t$ comes from central differences. The $u$ integrals cover $[0.001, 0.999]$ only. Both $\bar{\mathcal{A}}$ and $W_2^2$ are truncated to the same range, so the inequality and the identity hold for the truncated quantities as well.

In [ ]:
U_LO, U_HI, N_U = 0.001, 0.999, 1000
U_Q = U_LO + (U_HI - U_LO) * (np.arange(N_U) + 0.5) / N_U   # midpoint grid in [0.001, 0.999]
T_Q = T * np.linspace(0, 1, 801) ** 2                         # times, denser near t = 0
Q = np.stack([quantile_t(U_Q, t) for t in T_Q])              # Q[i, j] = F_{t_i}^{-1}(u_j)
dQ = np.gradient(Q, T_Q, axis=0)                              # central differences in t


def u_integral(values):
    """Integral over u in [U_LO, U_HI] of values on the midpoint grid (last axis)."""
    return (U_HI - U_LO) * np.mean(values, axis=-1)


A = np.trapezoid(u_integral(dQ**2), T_Q)                      # action in physical time
A_bar = T * A                                                 # action in normalized time s
W2_sq = (U_HI - U_LO) * w2_squared_between(Q[-1], Q[0])
geo_speed = (Q[-1] - Q[0]) / T
excess_identity_rhs = T * np.trapezoid(u_integral((dQ - geo_speed) ** 2), T_Q)


def action_density_x(t, n_x=4001):
    """Integral of p_t v_t^2 over [Q_t(U_LO), Q_t(U_HI)] with the exact velocity, trapezoid rule in x."""
    x = np.linspace(*quantile_t(np.array([U_LO, U_HI]), t), n_x)
    return np.trapezoid(p_t(x, t) * velocity(x, t) ** 2, x)


A_x = np.trapezoid([action_density_x(t) for t in T_Q], T_Q)   # independent of Q and dQ
print(f"A = {A:.4f} (quantile form)   A = {A_x:.4f} (x quadrature with the exact v_t)   "
      f"relative difference {abs(A - A_x) / A_x:.1e}")
print(f"A_bar = T A = {A_bar:.4f}   W_2^2(p_0, p_T) = {W2_sq:.4f}")
print(f"excess A_bar - W_2^2 = {A_bar - W2_sq:.4f}   right-hand side of the identity = {excess_identity_rhs:.4f}")

In [ ]:
# actions: the quantile form agrees with the x quadrature of p_t v_t^2 (relative tolerance 1e-4)
assert abs(A - A_x) < 1e-4 * A_x
assert A_bar >= W2_sq - 1e-6
assert abs((A_bar - W2_sq) - excess_identity_rhs) < 1e-3 * A_bar

The probability flow has $\bar{\mathcal{A}} = 5.78$, almost four times $W_2^2 = 1.50$. The quantile form of $\mathcal{A}$ agrees with the direct quadrature of $p_t v_t^2$ over $[Q_t(0.001), Q_t(0.999)]$ with the exact velocity, and the two sides of the identity agree to four digits. The figure shows where the excess comes from.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for u_show, color in zip((0.1, 0.3, 0.5, 0.7, 0.9), plt.cm.viridis(np.linspace(0, 0.9, 5))):
    j = int(np.argmin(np.abs(U_Q - u_show)))
    ax.plot(T_Q, dQ[:, j], color=color, lw=1.5, label=f"u = {U_Q[j]:.2f}")
    ax.axhline(geo_speed[j], color=color, lw=1.4, ls="--")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlim(0, T)
ax.set_xlabel("t")
ax.set_ylabel(r"$\partial_t Q_t(u)$")
ax.set_title("Quantile speed along the probability flow (solid) and the geodesic (dashed)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

*Speed of five quantiles along the probability flow (solid) and along the geodesic (dashed, same color). The flow moves each quantile fast at the start and has almost stopped by $t = 2$. The geodesic spreads the same displacement evenly over $[0, T]$.*

The quantiles move with the velocity $v_t$, and $v_t$ follows the changes in $p_t$. The component means decay like $e^{-t}$, and the component variances change fastest near $t = 0$, most of all for the narrow components. Both effects concentrate the motion near $t = 0$. The squared speed penalizes an uneven schedule, as the $h(t) = t^2$ schedule of the dynamic transport section did, so the front-loaded flow pays almost four times the geodesic cost for the same endpoint map.

In [ ]:
LEVELS = np.linspace(0.02, 0.98, 25)
U_F = (np.arange(20000) + 0.5) / 20000
q0, qT = quantile_t(U_F, 0.0), quantile_t(U_F, T)
inv0, invT = 1 / p_t(q0, 0.0), 1 / p_t(qT, T)                 # dQ_0/du and dQ_T/du
p_geo = np.empty((len(x_plot), len(T_SIM)))
for i, t in enumerate(T_SIM):
    s = t / T
    p_geo[:, i] = np.interp(x_plot, (1 - s) * q0 + s * qT, 1 / ((1 - s) * inv0 + s * invT),
                            left=0.0, right=0.0)
lev0, levT = quantile_t(LEVELS, 0.0), quantile_t(LEVELS, T)
geo_traj = (1 - T_SIM / T)[:, None] * lev0 + (T_SIM / T)[:, None] * levT
pf_traj = np.stack([quantile_t(LEVELS, t) for t in T_SIM])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
plot_transport(axes[0], T_SIM, x_plot, p_grid, trajectories=pf_traj,
               title="Probability flow", action=A_bar)
plot_transport(axes[1], T_SIM, x_plot, p_geo, trajectories=geo_traj,
               title="Displacement interpolation", action=W2_sq)
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

*Left, the probability flow path with quantile trajectories at 25 levels. Right, displacement interpolation between $p_0$ and $p_T$, with straight trajectories. Both titles report the normalized action: $\bar{\mathcal{A}}$ on the left and $W_2^2$ on the right. The flow merges the three components before $t = 1.5$. The geodesic keeps them apart until close to $t = T$.*

## From velocity to free energy

Diffusion does not take the geodesic, so the principle of least action does not explain the path it follows. The velocity $v_t = -x - \partial_x \log p_t$ appears again in the [next section](/aiml-common/lectures/diffusion/gradient-flow/gradient_flow), as the gradient of a free energy. That section shows that the Ornstein-Uhlenbeck path is the steepest descent of that free energy in the Wasserstein geometry.

## Numerical checks

The checks below list the integrated Fokker-Planck residual and the mass at each check time, and the $W_1$ distances for the three values of $\lambda$ beside their sampling baseline. They also compare the finite-difference quantile speed with the exact velocity $v_t(Q_t)$ away from $t = 0$, and give the path-length split. With $L(u) = \int_0^T |\partial_t Q_t(u)|\,dt$ the length of the path of quantile $u$, the Cauchy-Schwarz inequality in $t$ and then the triangle inequality give

$$
\bar{\mathcal{A}} \ge \int_0^1 L(u)^2\,du \ge W_2^2 .
$$

The first gap is positive when a quantile moves at uneven speed. The second is positive when a quantile reverses direction along the way.

In [ ]:
print("Fokker-Planck residual, integral of |dp_t/dt - RHS| dx, and mass:")
for t in T_CHECK + (T,):
    print(f"  t = {t:3.1f}   residual {fp_residual_l1(t):.1e}   mass {mass(t):.8f}")
print("\nW_1 of simulated particles to the exact p_t (rows: lambda; columns: t)")
print("            " + "".join(f"t = {t:<7g}" for t in t_check_all))
for lam, row in zip(LAMBDAS, w1_table):
    print(f"  lam = {lam:3.1f}  " + "".join(f"{w:<11.4f}" for w in row))
print("  baseline   " + "".join(f"{w:<11.4f}" for w in w1_baseline))
dq_gap = np.max(np.abs(dQ - np.stack([velocity(Q[i], t) for i, t in enumerate(T_Q)]))[T_Q >= 0.01])
print(f"\nlargest |dQ/dt (differences) - v_t(Q_t) (exact)| for t >= 0.01: {dq_gap:.1e}")
L = np.trapezoid(np.abs(dQ), T_Q, axis=0)                     # path length of each quantile
EL2 = u_integral(L**2)
print(f"path-length split: A_bar = {A_bar:.4f} >= E[L^2] = {EL2:.4f} >= W_2^2 = {W2_sq:.4f}")

In [ ]:
# quantiles move with the probability flow velocity: d/dt Q_t(u) = v_t(Q_t(u))
assert dq_gap < 1e-3
assert A_bar >= EL2 - 1e-9 and EL2 >= W2_sq - 1e-9

The residual, equilibrium and endpoint-map comparisons are repeated on a second mixture with weights $(0.5, 0.5)$, means $(-1.5, 2)$ and standard deviations $(0.5, 0.5)$.

In [ ]:
PI2, MU2, SIGMA2 = np.array([0.5, 0.5]), np.array([-1.5, 2.0]), np.array([0.5, 0.5])
MIX2 = dict(pi=PI2, mu=MU2, sigma=SIGMA2)
res2 = max(fp_residual_l1(t, **MIX2) for t in T_CHECK + (T,))
eq2 = np.max(np.abs(velocity(xs, 40.0, **MIX2)))
map2 = np.max(np.abs(pf_endpoint(u_mid, **MIX2) - quantile_t(u_mid, T, **MIX2)))
print(f"second mixture: largest residual {res2:.1e}   largest |v| at t = 40 {eq2:.1e}   "
      f"endpoint-map error {map2:.1e}")

In [ ]:
# the residual, equilibrium and endpoint-map checks on a second mixture
assert max(fp_residual_l1(t, **MIX2) for t in T_CHECK + (T,)) < 1e-6
assert max(abs(mass(t, **MIX2) - 1) for t in T_CHECK + (T,)) < 1e-6
assert np.max(np.abs(velocity(xs, 40.0, **MIX2))) < 1e-12
assert np.max(np.abs(pf_endpoint(u_mid, **MIX2) - quantile_t(u_mid, T, **MIX2))) < 1e-4

## References